# Non-linear PLSR: A Worked Example on the UCI Concrete Compressive Strength Dataset

Partial Least Squares Regression (PLSR) is a natural choice when predictors are numerous
and highly correlated — but its core assumption of a *linear* relationship between
predictors and response can break down in practice. This notebook walks through a concrete
(pun intended) example of when and how non-linear extensions of PLSR help.

We use the **UCI Concrete Compressive Strength dataset** (Yeh, 1998): 1,030 concrete mix
designs described by 8 ingredient/age variables, with the goal of predicting compressive
strength (MPa). The relationship between these ingredients and strength is well known to be
strongly non-linear — UCI's own dataset description notes that "the concrete compressive
strength is a highly nonlinear function of age and ingredients" — making it a good
illustrative case for comparing standard PLSR against a non-linear PLSR variant.

**Structure of this notebook:**
1. Load and explore the data
2. Demonstrate the non-linearity (EDA)
3. Fit standard PLSR as a baseline
4. Fit a non-linear PLSR model and compare performance
5. Key takeaways and practical considerations

In [2]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
concrete = fetch_ucirepo(id=165)  # Concrete Compressive Strength
X = concrete.data.features
y = concrete.data.targets

df = X.copy()
df["Strength"] = y

print(df.shape)
df.head()

(1030, 9)


,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


# Detecting Nonlinearity Before Fitting a Nonlinear PLSR Model -->

Before reaching for a nonlinear PLSR variant, it is good practice to first show — not just assume — that the standard linear PLS assumption is actually violated. Three complementary, well-established diagnostics are used here.

1. Latent score vs. response (the PLS-specific diagnostic)

Standard PLSR assumes a linear inner relation: the response y is a linear function of the latent scores t extracted from the predictors. Plotting the first latent score t1 against y and comparing it to a smoothed trend (LOWESS) reveals whether this core assumption holds. Systematic curvature in this plot is precisely the diagnostic that motivated Wold's original development of nonlinear PLS (quadratic PLS, spline PLS): if the inner relation is not a straight line, forcing a linear model onto it discards structure the model could otherwise capture. This makes it the most directly relevant diagnostic for justifying a non-linear PLSR approach, since it targets the exact assumption non-linear PLS relaxes.

2. Residuals vs. fitted values (standard regression diagnostic)

A textbook diagnostic (Draper & Smith, Applied Regression Analysis): under a correctly specified model, residuals should scatter randomly around zero with no pattern. A curved band in the residual plot indicates unmodelled structure — typically a sign of nonlinearity the model has not captured.

3. Ramsey's RESET test (formal statistical test)

Ramsey's Regression Equation Specification Error Test (Ramsey, 1969, Journal of the Royal Statistical Society, Series B) is a widely cited, formal hypothesis test for functional-form misspecification. It tests whether adding higher-order powers of the fitted values (here: squared and cubed) to the model significantly improves fit. A significant result (low p-value) is evidence that the linear model omits real nonlinear structure. It is a standard tool in econometrics and applied statistics for exactly this question, and applies directly here because it only requires a set of fitted values and a response — it does not care whether those fitted values came from OLS or from PLS.

## Result on the Concrete Compressive Strength dataset

Running all three diagnostics on a linear PLS baseline (component count chosen via 10-fold cross-validation) gives a consistent picture:

The t1 vs. y plot shows clear curvature relative to the linear inner relation, most visible at the extremes of the score range.
The residual-vs-fitted plot shows a non-random, curved pattern rather than an even scatter around zero.
The RESET test rejects the null hypothesis of correct linear specification (F ≈ 19.4, p ≈ 5×10⁻⁹), i.e. there is strong statistical evidence of nonlinearity that the linear PLS model fails to capture.

Together, these three checks — one PLS-specific, one general regression diagnostic, and one formal statistical test — build a solid, citable case for moving to a non-linear PLSR approach on this dataset.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("pgf")
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
import statsmodels.api as sm
import shutil

print(shutil.which('pdflatex'))

from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------
# Beamer font matching
# ---------------------------------------------------------------------
BEAMER_FONTSIZE = 11
BEAMER_FONT_FAMILY = "sans-serif"
matplotlib.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "font.family": BEAMER_FONT_FAMILY,
    "font.size": BEAMER_FONTSIZE,
    "axes.labelsize": BEAMER_FONTSIZE,
    "xtick.labelsize": BEAMER_FONTSIZE - 1,
    "ytick.labelsize": BEAMER_FONTSIZE - 1,
    "legend.fontsize": BEAMER_FONTSIZE - 1,
    "pgf.rcfonts": False,
})


C:\Users\jakob\AppData\Local\Programs\MiKTeX\miktex\bin\x64\pdflatex.EXE


In [ ]:

# ---------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------

df.columns = [c.strip() for c in df.columns]
# df = df.rename(columns={"Concrete compressive strength": "Strength"})

X = df.drop(columns="Strength").values
y = df["Strength"].values

X_scaled = StandardScaler().fit_transform(X)
y_scaled = StandardScaler().fit_transform(y.reshape(-1, 1)).ravel()

# ---------------------------------------------------------------------
# 2. Select number of components for the LINEAR PLS baseline via
#    10-fold CV (RMSECV), to get a fair baseline before checking
#    whether it is well-specified.
# ---------------------------------------------------------------------
cv = KFold(n_splits=10, shuffle=True, random_state=42)
rmsecv = []
for ncomp in range(1, 9):
    pls = PLSRegression(n_components=ncomp)
    y_cv = cross_val_predict(pls, X_scaled, y_scaled, cv=cv)
    rmsecv.append(np.sqrt(np.mean((y_scaled - y_cv) ** 2)))

best_ncomp = int(np.argmin(rmsecv)) + 1
print(f"Selected {best_ncomp} PLS components (lowest RMSECV = {min(rmsecv):.3f})")





Selected 7 PLS components (lowest RMSECV = 0.627)


In [ ]:
# ---------------------------------------------------------------------
# 3. Fit the linear PLS baseline on the full data
# ---------------------------------------------------------------------
pls = PLSRegression(n_components=best_ncomp)
pls.fit(X_scaled, y_scaled)

y_fit = pls.predict(X_scaled).ravel()
t1 = pls.x_scores_[:, 0]  # first latent variable (score)

residuals = y_scaled - y_fit


# ---------------------------------------------------------------------
# 4. Diagnostic 1: first latent score vs. response
#
#    LOWESS shows the local relationship between the first latent
#    score t1 and the response.
#
#    If the LOWESS curve shows clear curvature, this may indicate
#    that the linear inner relation in PLS does not describe the
#    relationship adequately.
# ---------------------------------------------------------------------
smoothed_t1 = lowess(
    y_scaled,
    t1,
    frac=0.4,
    return_sorted=True
)

# Figure size chosen to fit one half of a 16:9 Beamer slide
fig1, ax1 = plt.subplots(figsize=(2.45, 1.75))

# Observations
ax1.scatter(
    t1,
    y_scaled,
    s=7,
    alpha=0.35,
    color="#4C72B0",
    rasterized=False
)

# LOWESS trend
ax1.plot(
    smoothed_t1[:, 0],
    smoothed_t1[:, 1],
    color="#C44E52",
    linewidth=1.8,
    label="LOWESS"
)

# Linear reference relation
slope, intercept = np.polyfit(t1, y_scaled, 1)

ax1.axline(
    (0, intercept),
    slope=slope,
    color="grey",
    linestyle="--",
    linewidth=1.1,
    label="Linear relation"
)

# Axis labels
ax1.set_xlabel(r"First PLS latent score $t_1$")
ax1.set_ylabel("Standardized strength")

# Clean appearance
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax1.tick_params(
    axis="both",
    which="major",
    pad=2
)

# Legend without frame
ax1.legend(
    frameon=False,
    loc="best",
    fontsize=8
)

# Leave enough space for axis labels
fig1.subplots_adjust(
    left=0.18,
    right=0.98,
    bottom=0.23,
    top=0.97
)

# Save as PGF for LaTeX/Beamer and PDF for preview
fig1.savefig("t1_vs_y.pgf")
fig1.savefig("t1_vs_y.pdf")

plt.close(fig1)

In [ ]:
# ---------------------------------------------------------------------
# 5. Diagnostic 2: residuals vs. fitted values
#
#    The residual plot is used to look for systematic patterns that
#    are not captured by the linear PLS model.
#
#    A LOWESS curve that stays approximately around zero supports the
#    linear specification visually. Clear curvature or systematic
#    deviations from zero may indicate unmodelled nonlinear structure.
# ---------------------------------------------------------------------
smoothed_resid = lowess(
    residuals,
    y_fit,
    frac=0.4,
    return_sorted=True
)

# Same physical size as the first diagnostic figure
fig2, ax2 = plt.subplots(figsize=(2.45, 1.75))

# Residuals
ax2.scatter(
    y_fit,
    residuals,
    s=7,
    alpha=0.35,
    color="#4C72B0",
    rasterized=False
)

# LOWESS trend
ax2.plot(
    smoothed_resid[:, 0],
    smoothed_resid[:, 1],
    color="#C44E52",
    linewidth=1.8,
    label="LOWESS"
)

# Zero-reference line
ax2.axhline(
    0,
    color="grey",
    linestyle="--",
    linewidth=1.1
)

# Axis labels
ax2.set_xlabel("Fitted values (linear PLS)")
ax2.set_ylabel("Residuals")

# Clean appearance
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

ax2.tick_params(
    axis="both",
    which="major",
    pad=2
)

# Legend without frame
ax2.legend(
    frameon=False,
    loc="best",
    fontsize=8
)

# Leave enough space for axis labels
fig2.subplots_adjust(
    left=0.18,
    right=0.98,
    bottom=0.23,
    top=0.97
)

# Save as PGF for LaTeX/Beamer and PDF for preview
fig2.savefig("residuals_vs_fitted.pgf")
fig2.savefig("residuals_vs_fitted.pdf")

plt.close(fig2)


# ---------------------------------------------------------------------
# 6. Diagnostic 3: Ramsey's RESET test
#
#    H0: the linear PLS fit is correctly specified with respect to
#        the functional form tested here.
#
#    The unrestricted model adds squared and cubed fitted values:
#
#        y = beta_0 + beta_1*y_hat + beta_2*y_hat^2
#            + beta_3*y_hat^3 + error
#
#    A small p-value provides statistical evidence that the additional
#    nonlinear terms improve the specification.
# ---------------------------------------------------------------------
yhat = y_fit.reshape(-1, 1)

# Restricted model: linear relationship
X_restricted = sm.add_constant(yhat)

# Full model: linear + quadratic + cubic terms
X_full = sm.add_constant(
    np.column_stack([
        yhat,
        yhat**2,
        yhat**3
    ])
)

# Fit both models
model_restricted = sm.OLS(
    y_scaled,
    X_restricted
).fit()

model_full = sm.OLS(
    y_scaled,
    X_full
).fit()

# Compare restricted and unrestricted models
reset_result = model_full.compare_f_test(
    model_restricted
)

f_stat, p_value, df_diff = reset_result

# Print RESET results
print("\nRamsey RESET test on the linear PLS fit")
print(f"  F-statistic = {f_stat:.3f}")
print(f"  p-value     = {p_value:.2e}")
print(f"  df difference = {df_diff}")

if p_value < 0.05:
    print(
        "  -> Reject H0: evidence of nonlinear structure "
        "not captured by the linear PLS model."
    )
else:
    print(
        "  -> Fail to reject H0: no strong statistical evidence "
        "of nonlinear structure."
    )


Ramsey RESET test on the linear PLS fit
  F-statistic = 19.436
  p-value     = 5.19e-09
  df difference = 2.0
  -> Reject H0: evidence of nonlinear structure not captured by the linear PLS model.


1. Latent Score
PLS takes the original variables and combines them into a smaller number of latent variables.
The first latent score, (t_1), is a weighted combination of the original predictors.
It summarizes the main direction of variation in the predictor data.
We plot (t_1) against the response, concrete strength.
A straight relationship would support the linear PLS assumption.
2. LOWESS
The red LOWESS curve shows the local trend in the data.
LOWESS means Locally Weighted Scatterplot Smoothing.
It does not force the relationship to be a straight line.
This makes it useful for checking whether the relationship has curvature.
If the LOWESS curve bends away from a straight line, this is evidence of a nonlinear relationship.
3. Residuals
A residual is the difference between the actual value and the model's prediction:

[
e_i = y_i - \hat{y}_i
]

Positive residual: the model predicted too low.
Negative residual: the model predicted too high.
If the linear model is appropriate, residuals should look roughly randomly scattered around zero.
A clear pattern or curve means the model is systematically missing something.
4. RESET Test
The RESET test gives us a formal statistical check of the functional form.
We start with the linear PLS predictions, (\hat y).
Then we add nonlinear terms such as:

[
\hat y^2,\quad \hat y^3
]

The null hypothesis is that the original linear specification is adequate.
A small p-value means that the additional nonlinear terms provide evidence that the linear specification is not adequate.
5. What Do We See?
Latent Score Plot
The points show the observations.
The dashed line represents the linear relationship.
The LOWESS curve bends relative to the linear relationship.
This suggests that the relationship between the latent score and strength is not fully linear.
Residual Plot
Ideally, the residuals should form a random cloud around zero.
Here, the LOWESS curve shows systematic curvature.
This suggests that the linear PLS model leaves some structure unexplained.
6. Overall Conclusion
The two graphical diagnostics both show signs of systematic curvature.
RESET provides a formal statistical check of the same issue.
Taken together, the diagnostics suggest that the linear PLS specification does not capture the full relationship.
This motivates looking at a nonlinear extension of the PLS model.

# Kernel PLS: choose the kernel and compare nonlinear models

The linear PLS baseline already shows clear curvature, so the next step is to compare nonlinear kernels before choosing the final model.

A practical rule for this kind of dataset is:

- Start with an RBF kernel; it is usually the best first choice for smooth nonlinear relationships.
- Add a polynomial kernel as a comparison when the relation may contain low-order curvature or interaction effects.
- Use cross-validation to compare kernel choices, not just the training fit.

This workflow is the standard way to choose a nonlinear PLS specification in practice, because scikit-learn does not provide a native `KernelPLS` estimator. Instead, we compare kernelized feature maps and then fit PLS on the mapped representation.

## Recommended workflow
1. Standardize the predictors and response.
2. Build a kernel feature map using either RBF or polynomial kernels.
3. Fit a PLS model on the transformed features.
4. Compare cross-validated RMSE across kernel types and parameter values.
5. Pick the simplest model with the lowest RMSECV.

## Interpretation
- If the RBF kernel gives the smallest RMSECV, it usually means the relationship is smooth and nonlinear.
- If a polynomial kernel performs similarly or better, the dependence may be dominated by curvature and interaction-type effects.
- If several models are close, choose the simpler one and check the residual pattern again.

This step is important because the kernel choice is not just a technical detail: it determines how the latent structure is extracted and how well the model captures the nonlinear relation between the ingredients and concrete strength.


In [5]:
from sklearn.decomposition import KernelPCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import numpy as np


def kernel_pls_cv_score(
    X,
    y,
    kernel="rbf",
    gamma=1.0,
    degree=2,
    n_components=2,
    n_splits=5,
    random_state=42,
):
    """Cross-validated RMSE for a kernelized-feature + PLS workflow."""
    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    y_pred = np.zeros_like(y, dtype=float)

    for train_idx, valid_idx in cv.split(X):
        X_train, X_valid = X[train_idx], X[valid_idx]
        y_train = y[train_idx]

        kpca = KernelPCA(
            n_components=min(10, X_train.shape[0] - 1),
            kernel=kernel,
            gamma=gamma,
            degree=degree,
            fit_inverse_transform=False,
        )

        X_train_k = kpca.fit_transform(X_train)
        X_valid_k = kpca.transform(X_valid)

        pls = PLSRegression(n_components=min(n_components, X_train_k.shape[1]))
        pls.fit(X_train_k, y_train)
        y_pred[valid_idx] = pls.predict(X_valid_k).ravel()

    rmse = np.sqrt(mean_squared_error(y, y_pred))
    return rmse


# ---------------------------------------------------------------------
# 1. Prepare the data
# ---------------------------------------------------------------------
if "X_scaled" not in globals() or "y_scaled" not in globals():
    df.columns = [c.strip() for c in df.columns]
    X_array = df.drop(columns="Strength").values
    y_array = df["Strength"].values
    X_scaled = StandardScaler().fit_transform(X_array)
    y_scaled = StandardScaler().fit_transform(y_array.reshape(-1, 1)).ravel()

X_model = X_scaled
y_model = y_scaled

# ---------------------------------------------------------------------
# 2. Compare kernel choices with cross-validation
# ---------------------------------------------------------------------
results = []
for kernel in ["rbf", "poly"]:
    if kernel == "rbf":
        gammas = [0.01, 0.05, 0.1, 0.5, 1.0]
        degrees = [None]
    else:
        gammas = [0.01, 0.05, 0.1, 0.5, 1.0]
        degrees = [2, 3]

    for gamma in gammas:
        for degree in degrees:
            for ncomp in [2, 3, 4, 5, 6, 7]:
                rmse = kernel_pls_cv_score(
                    X_model,
                    y_model,
                    kernel=kernel,
                    gamma=gamma,
                    degree=degree if degree is not None else 2,
                    n_components=ncomp,
                    n_splits=5,
                    random_state=42,
                )

                results.append(
                    {
                        "kernel": kernel,
                        "gamma": gamma,
                        "degree": degree,
                        "n_components": ncomp,
                        "rmse_cv": rmse,
                    }
                )

results = sorted(results, key=lambda d: d["rmse_cv"])

print("Top kernel-PLS candidates by 5-fold RMSECV:")
for r in results[:10]:
    deg = "-" if r["degree"] is None else r["degree"]
    print(
        f"kernel={r['kernel']:>4s}, gamma={r['gamma']:.2f}, "
        f"degree={deg:>4s}, ncomp={r['n_components']:>2d}, "
        f"RMSECV={r['rmse_cv']:.4f}"
    )

best = results[0]
print("\nBest starting choice:")
print(f"  kernel = {best['kernel']}")
print(f"  gamma  = {best['gamma']}")
print(f"  degree = {best['degree']}")
print(f"  ncomp  = {best['n_components']}")
print(f"  RMSECV = {best['rmse_cv']:.4f}")

# ---------------------------------------------------------------------
# 3. Fit the final model using the best kernel choice
# ---------------------------------------------------------------------
kpca = KernelPCA(
    n_components=min(10, X_model.shape[0] - 1),
    kernel=best["kernel"],
    gamma=best["gamma"],
    degree=best["degree"] if best["degree"] is not None else 2,
    fit_inverse_transform=False,
)
X_k = kpca.fit_transform(X_model)

pls_final = PLSRegression(n_components=best["n_components"])
pls_final.fit(X_k, y_model)

y_fit = pls_final.predict(X_k).ravel()
train_rmse = np.sqrt(np.mean((y_model - y_fit) ** 2))
print(f"\nFinal training RMSE = {train_rmse:.4f}")

# ---------------------------------------------------------------------
# 4. Practical interpretation
# ---------------------------------------------------------------------
print("\nInterpretation:")
print("- RBF is usually the safest first choice for smooth nonlinear relationships.")
print("- Polynomial is useful if the data appear to follow curvature or interaction patterns.")
print("- The selected kernel should be the one with the lowest RMSECV, not the best training fit.")


c:\Users\jakob\miniconda3\envs\TK8117\Lib\site-packages\sklearn\cross_decomposition\_pls.py:325: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(yk, y_weights) / y_ss
c:\Users\jakob\miniconda3\envs\TK8117\Lib\site-packages\sklearn\cross_decomposition\_pls.py:325: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(yk, y_weights) / y_ss
c:\Users\jakob\miniconda3\envs\TK8117\Lib\site-packages\sklearn\cross_decomposition\_pls.py:325: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(yk, y_weights) / y_ss
c:\Users\jakob\miniconda3\envs\TK8117\Lib\site-packages\sklearn\cross_decomposition\_pls.py:325: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(yk, y_weights) / y_ss
c:\Users\jakob\miniconda3\envs\TK8117\Lib\site-packages\sklearn\cross_decomposition\_pls.py:325: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(yk, y_weights) / y_ss
c:\Users\jakob\miniconda3\envs\TK8117\Lib\sit

Top kernel-PLS candidates by 5-fold RMSECV:
kernel= rbf, gamma=0.01, degree=   -, ncomp= 2, RMSECV=0.5871
kernel= rbf, gamma=0.01, degree=   -, ncomp= 3, RMSECV=0.5871
kernel= rbf, gamma=0.01, degree=   -, ncomp= 5, RMSECV=0.5871
kernel= rbf, gamma=0.01, degree=   -, ncomp= 6, RMSECV=0.5871
kernel= rbf, gamma=0.01, degree=   -, ncomp= 4, RMSECV=0.5871
kernel= rbf, gamma=0.01, degree=   -, ncomp= 7, RMSECV=0.5871
kernel= rbf, gamma=0.05, degree=   -, ncomp= 2, RMSECV=0.6094
kernel= rbf, gamma=0.05, degree=   -, ncomp= 3, RMSECV=0.6094
kernel= rbf, gamma=0.05, degree=   -, ncomp= 4, RMSECV=0.6094
kernel= rbf, gamma=0.05, degree=   -, ncomp= 5, RMSECV=0.6094

Best starting choice:
  kernel = rbf
  gamma  = 0.01
  degree = None
  ncomp  = 2
  RMSECV = 0.5871

Final training RMSE = 0.5726

Interpretation:
- RBF is usually the safest first choice for smooth nonlinear relationships.
- Polynomial is useful if the data appear to follow curvature or interaction patterns.
- The selected kernel sh

c:\Users\jakob\miniconda3\envs\TK8117\Lib\site-packages\sklearn\cross_decomposition\_pls.py:325: RuntimeWarning: invalid value encountered in divide
  y_scores = np.dot(yk, y_weights) / y_ss
